# GPU 01 - OCT-F Fine-Tuning, Residual Cache and Mask Extraction

Fine-tunes the OCT-F foundation model on baseline QC scans, caches residuals for the 74 annotated endpoint images, and extracts the binary lesion masks.

## 1. Setup

Paths are relative to the project root. Seed is 40494109 for every split and shuffle.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pickle, time

# project root (notebook runs from gpu_notebooks/)
PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))

# raw data
ENDPOINT_FOLDER = os.path.join(PROJECT_ROOT, 'data',
                    'endpoint')
QC_FOLDERS = [os.path.join(PROJECT_ROOT, 'data', 'baseline_qc'),
              os.path.join(PROJECT_ROOT, 'data', 'baseline_qc_cohort_b')]
ANNO_FOLDER = os.path.join(PROJECT_ROOT, 'data', 'annotated',
               'endpoint_annotated')

# generated data
MASK_FOLDER = os.path.join(PROJECT_ROOT, 'data', 'masks')

# reconstruction
OCTF_CODE = os.path.join(PROJECT_ROOT, 'reconstruction',
             'OCT-F_Reconstruction', 'code')
ORIGINAL_CKPT = os.path.join(PROJECT_ROOT, 'reconstruction',
                 'OCT-F_Reconstruction', 'checkpoint',
                 'model_epoch_38_loss_0.1931.pth')
FINETUNE_CKPT = os.path.join(PROJECT_ROOT, 'reconstruction',
                 'fine_tuned_reconstruction.pth')
RESIDUAL_CACHE = os.path.join(PROJECT_ROOT, 'reconstruction',
                  'residual_cache.pkl')
CKPT_DIR = os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints')

# shared code first so code/datasets.py and code/losses.py are not
# shadowed by the OCT-F files of the same name
sys.path.insert(0, OCTF_CODE)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

# outputs
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
CURVES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'training_curves')
IMAGES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'images')

for d in [MASK_FOLDER, CKPT_DIR, FIGURES_DIR, CURVES_DIR, IMAGES_DIR]:
    os.makedirs(d, exist_ok=True)

# single seed for every split, shuffle and weight initialisation
SEED = 40494109
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

IMG_SIZE = 512

PP_KERNEL = int(15 * IMG_SIZE / 256)
PP_KERNEL = PP_KERNEL if PP_KERNEL % 2 == 1 else PP_KERNEL - 1
EROSION_KERNEL = int(5 * IMG_SIZE / 256)
EROSION_KERNEL = EROSION_KERNEL if EROSION_KERNEL % 2 == 1 else EROSION_KERNEL - 1
print(f"Image size: {IMG_SIZE} | PP kernel: {PP_KERNEL} | Erosion kernel: {EROSION_KERNEL}")

for path, name in [(ENDPOINT_FOLDER, 'Endpoint images'),
                   (QC_FOLDERS[0], 'QC scans (cohort A)'),
                   (QC_FOLDERS[1], 'QC scans (cohort B)'),
                   (ANNO_FOLDER, 'Annotated images'),
                   (ORIGINAL_CKPT, 'OCT-F checkpoint')]:
    assert os.path.exists(path), f"MISSING: {name} at {path}"
    print(f"  Found: {name}")
print("All starting files verified.")

## 2. Pretrained OCT-F Model

The model reconstructs OCT B-scans at 496x768. Weights come from the NICOLA human retinal dataset.

In [ ]:
from architectureV2Res_new import ArchitectureV2

model = ArchitectureV2().to(device)

# checkpoint was saved under DataParallel, so the module. prefix is stripped
ckpt = torch.load(ORIGINAL_CKPT, map_location=device, weights_only=False)
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state_dict'].items()}
model.load_state_dict(state_dict)

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded OCT-F from epoch {ckpt['epoch']}, training loss {ckpt['loss']:.4f}")
print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")

## 3. QC Split

QC images are the per-eye baseline, taken immediately after lasering. Split 74 train / 19 validation, seed 40494109.

In [ ]:
# both studies contribute QC images, filtered on extension so
# the mis-named "a baseline QC scan" is not dropped
qc_paths = []
for folder in QC_FOLDERS:
    for f in sorted(os.listdir(folder)):
        if f.endswith('.tif'):
            qc_paths.append(os.path.join(folder, f))

print(f"QC images found: {len(qc_paths)}")
for folder in QC_FOLDERS:
    n = len([p for p in qc_paths if os.path.dirname(p) == folder])
    print(f"  {os.path.basename(folder)}: {n}")
assert len(qc_paths) == 93, f"expected 93 QC images, found {len(qc_paths)}"

np.random.seed(SEED)
np.random.shuffle(qc_paths)

qc_train = qc_paths[:74]
qc_val = qc_paths[74:]
print(f"\nTrain: {len(qc_train)}  Validation: {len(qc_val)}")
print("Validation set is held out and used only for SSIM reporting.")

## 4. QC Dataset

Images are resized to 496x768, the size OCT-F expects. Input and target are the same image.

In [ ]:
class BaselineQCDataset(Dataset):
    """QC images for reconstruction. Returns (image, image)."""

    def __init__(self, image_paths, augment=False):
        self.image_paths = image_paths
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # single channel extracted, images are 3-channel RGB with R=G=B
        img = np.array(Image.open(self.image_paths[idx]))
        gray = cv2.resize(img[:, :, 0], (768, 496)).astype(np.float32) / 255.0

        if self.augment and np.random.random() > 0.5:
            gray = np.fliplr(gray).copy()

        tensor = torch.from_numpy(gray).unsqueeze(0)
        return tensor, tensor


train_dataset = BaselineQCDataset(qc_train, augment=True)
val_dataset = BaselineQCDataset(qc_val, augment=False)

# batch size 2 keeps memory manageable at 496x768 and matches the earlier run
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

sample_input, sample_target = train_dataset[0]
print(f"Sample shape: {tuple(sample_input.shape)}  "
      f"range [{sample_input.min():.3f}, {sample_input.max():.3f}]")
print(f"Train batches: {len(train_loader)}  Validation batches: {len(val_loader)}")

## 5. Reconstruction Loss

Equally weighted MSE and SSIM, matching the OCT-F training objective.

In [ ]:
from pytorch_msssim import ssim


class ReconstructionLoss(nn.Module):
    """0.5 MSE + 0.5 (1 - SSIM). Returns (total, mse, ssim_value)."""

    def __init__(self, mse_weight=0.5, ssim_weight=0.5):
        super().__init__()
        self.mse_weight = mse_weight
        self.ssim_weight = ssim_weight
        self.mse = nn.MSELoss()

    def forward(self, reconstruction, original):
        mse_loss = self.mse(reconstruction, original)
        ssim_value = ssim(reconstruction, original, data_range=1.0, size_average=True)
        total = self.mse_weight * mse_loss + self.ssim_weight * (1 - ssim_value)
        return total, mse_loss.item(), ssim_value.item()


criterion = ReconstructionLoss()
print("Loss: 0.5 x MSE + 0.5 x (1 - SSIM)")

## 6. Baseline SSIM

SSIM on the held-out QC images before fine-tuning, for comparison after training.

In [ ]:
def evaluate_ssim(model, paths, device):
    """Per-image SSIM on held-out QC images."""
    model.eval()
    scores = []
    with torch.no_grad():
        for p in paths:
            img = np.array(Image.open(p))
            gray = cv2.resize(img[:, :, 0], (768, 496)).astype(np.float32) / 255.0
            x = torch.from_numpy(gray).unsqueeze(0).unsqueeze(0).to(device)
            recon, _ = model(x)
            # decoder returns 768x496, transposed back to match the input
            recon = recon.permute(0, 1, 3, 2)
            scores.append(ssim(recon, x, data_range=1.0, size_average=True).item())
    return float(np.mean(scores)), float(np.std(scores)), scores


baseline_ssim, baseline_std, _ = evaluate_ssim(model, qc_val, device)
print(f"Baseline SSIM on {len(qc_val)} held-out QC images: "
      f"{baseline_ssim:.4f} (std {baseline_std:.4f})")

## 7. Fine-Tuning

10 epochs, AdamW at lr 1e-5, cosine annealing, early stopping patience 10. Best checkpoint is kept by validation loss.

In [ ]:
NUM_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 1e-5

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS,
                                                 eta_min=1e-7)

history = {'train_loss': [], 'val_loss': [], 'train_ssim': [], 'val_ssim': [],
           'lr': [], 'epoch_time': []}

best_val_loss = float('inf')
best_val_ssim = 0.0
best_epoch = 0
no_improve = 0
stop_epoch = NUM_EPOCHS

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    epoch_lr = optimizer.param_groups[0]['lr']

    model.train()
    train_loss, train_ssim = 0.0, 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        reconstruction, latent = model(inputs)
        # decoder returns 768x496, transposed back to match the target
        reconstruction = reconstruction.permute(0, 1, 3, 2)
        loss, mse_val, ssim_val = criterion(reconstruction, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_ssim += ssim_val

    train_loss /= len(train_loader)
    train_ssim /= len(train_loader)

    model.eval()
    val_loss, val_ssim = 0.0, 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            reconstruction, latent = model(inputs)
            reconstruction = reconstruction.permute(0, 1, 3, 2)
            loss, mse_val, ssim_val = criterion(reconstruction, targets)
            val_loss += loss.item()
            val_ssim += ssim_val

    val_loss /= len(val_loader)
    val_ssim /= len(val_loader)

    scheduler.step()
    epoch_time = time.time() - epoch_start

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_ssim'].append(train_ssim)
    history['val_ssim'].append(val_ssim)
    history['lr'].append(epoch_lr)
    history['epoch_time'].append(epoch_time)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_ssim = val_ssim
        best_epoch = epoch
        no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        marker = " [BEST]"
    else:
        no_improve += 1
        marker = ""

    print(f"Epoch {epoch}/{NUM_EPOCHS} | Train: {train_loss:.4f} (SSIM {train_ssim:.4f}) | "
          f"Val: {val_loss:.4f} (SSIM {val_ssim:.4f}) | LR: {epoch_lr:.7f} | "
          f"{epoch_time:.1f}s{marker}")

    if no_improve >= PATIENCE:
        stop_epoch = epoch
        print(f"Early stopping at epoch {epoch}, no improvement for {PATIENCE} epochs")
        break

model.load_state_dict(best_state)
print(f"\nBest epoch {best_epoch} | val_loss {best_val_loss:.4f} | val_ssim {best_val_ssim:.4f}")

## 8. Held-Out SSIM

SSIM measured on the 19 validation QC images only. These were never used for weight updates.

In [ ]:
final_ssim, final_std, per_image_ssim = evaluate_ssim(model, qc_val, device)

print(f"{'Stage':<20} {'SSIM':<10} {'Std':<10}")
print("=" * 40)
print(f"{'Before fine-tuning':<20} {baseline_ssim:<10.4f} {baseline_std:<10.4f}")
print(f"{'After fine-tuning':<20} {final_ssim:<10.4f} {final_std:<10.4f}")
print(f"\nMeasured on {len(qc_val)} held-out QC images, none seen during training.")

## 9. Save Fine-Tuned Model

Checkpoint carries the state dict, full history and config for notebook 02.

In [ ]:
torch.save({
    'model_name': 'OCT-F fine-tuned on baseline QC scans',
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'val_loss': best_val_loss,
    'val_ssim': best_val_ssim,
    'held_out_ssim': final_ssim,
    'held_out_ssim_std': final_std,
    'baseline_ssim': baseline_ssim,
    'baseline_ssim_std': baseline_std,
    'per_image_ssim': per_image_ssim,
    'history': history,
    'config': {
        'img_size': [496, 768],
        'batch_size': 2,
        'lr': LEARNING_RATE,
        'weight_decay': 1e-4,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'loss_function': 'ReconstructionLoss',
        'max_epochs': NUM_EPOCHS,
        'early_stopped_at': stop_epoch,
        'patience': PATIENCE,
        'seed': SEED,
        'qc_train': len(qc_train),
        'qc_val': len(qc_val),
        'params': total_params,
    },
    'qc_train_files': [os.path.basename(p) for p in qc_train],
    'qc_val_files': [os.path.basename(p) for p in qc_val],
}, FINETUNE_CKPT)

size_mb = os.path.getsize(FINETUNE_CKPT) / 1e6
print(f"Saved {FINETUNE_CKPT} ({size_mb:.1f} MB)")

## 10. Residual Cache

Residuals are `|original - reconstruction|` at 496x768. The encoder concatenates a random tensor, so the seed is reset first to keep the cache reproducible.

In [ ]:
def reconstruct_with_model(image_gray, model, device):
    """Return (original, reconstruction, residual) at 496x768."""
    resized = cv2.resize(image_gray, (768, 496))
    x = torch.from_numpy(resized).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        reconstruction, latent = model(x)
    original = x.squeeze().cpu().numpy()
    # decoder returns 768x496, transposed back to match the original
    reconstruction = reconstruction.squeeze().cpu().numpy().T
    residual = np.abs(original - reconstruction)
    return original, reconstruction, residual


model.eval()
torch.manual_seed(SEED)

anno_files = sorted([f for f in os.listdir(ANNO_FOLDER) if f.endswith('.tif')])
residual_cache = {}
start = time.time()

for filename in anno_files:
    img = np.array(Image.open(os.path.join(ENDPOINT_FOLDER, filename)))
    _, _, residual = reconstruct_with_model(img[:, :, 0], model, device)
    residual_cache[filename] = residual.astype(np.float32)

with open(RESIDUAL_CACHE, 'wb') as f:
    pickle.dump(residual_cache, f)

shapes = {r.shape for r in residual_cache.values()}
values = np.concatenate([r.ravel() for r in residual_cache.values()])
print(f"Cached {len(residual_cache)} residuals in {time.time() - start:.1f}s")
print(f"Shapes: {shapes}")
print(f"Residual range: [{values.min():.4f}, {values.max():.4f}]  mean {values.mean():.4f}")
print(f"Saved {RESIDUAL_CACHE} ({os.path.getsize(RESIDUAL_CACHE) / 1e6:.1f} MB)")

## 11. Lesion Mask Extraction

Yellow annotation outlines are thresholded and filled to produce binary masks at the original 494x765.

In [ ]:
mask_files = []
coverage = []

for filename in anno_files:
    anno = np.array(Image.open(os.path.join(ANNO_FOLDER, filename)))

    R = anno[:, :, 0].astype(float)
    G = anno[:, :, 1].astype(float)
    B = anno[:, :, 2].astype(float)
    yellow = ((R > 200) & (G > 200) & (B < 100)).astype(np.uint8)

    # the annotation is an outline, filled here to give a solid region
    contours, _ = cv2.findContours(yellow, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(yellow)
    cv2.drawContours(filled, contours, -1, 1, thickness=cv2.FILLED)

    assert filled.sum() > 0, f"no annotation found in {filename}"
    cv2.imwrite(os.path.join(MASK_FOLDER, filename.replace('.tif', '_mask.png')),
                filled * 255)
    mask_files.append(filename)
    coverage.append(filled.mean() * 100)

print(f"Masks written: {len(mask_files)} / {len(anno_files)}")
print(f"Lesion coverage: min {min(coverage):.2f}%  mean {np.mean(coverage):.2f}%  "
      f"max {max(coverage):.2f}%")
print(f"Saved to {MASK_FOLDER}")

## 12. Outputs

All three artefacts required by gpu_02 and the analysis notebooks.

In [ ]:
outputs = [
    ('fine_tuned_reconstruction.pth', FINETUNE_CKPT, None),
    ('residual_cache.pkl', RESIDUAL_CACHE, 74),
    ('data/masks/', MASK_FOLDER, 74),
]

print(f"{'Output':<28} {'Status':<10} {'Count':<8} {'Size'}")
print("=" * 60)
for name, path, expected in outputs:
    exists = os.path.exists(path)
    if os.path.isdir(path):
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        size = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) / 1e6
    else:
        count = expected if expected else '-'
        size = os.path.getsize(path) / 1e6 if exists else 0
    status = 'OK' if exists and (expected is None or count == expected) else 'CHECK'
    print(f"{name:<28} {status:<10} {str(count):<8} {size:.1f} MB")

print(f"\nHeld-out SSIM: {baseline_ssim:.4f} -> {final_ssim:.4f}")
print("Next: run gpu_02_train_all.ipynb")